## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2") # uses 384-d vectors

In [8]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [9]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [10]:
v1.dot(dv)

np.float32(0.32332397)

In [11]:
# unrelated query
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [12]:
v2.dot(dv)

np.float32(0.019730574)

Notice 0.32 vs 0.01. The closer to 0 means the vectors are orthagonal and very different.

## Cosine Similarity

`all-MiniLM-L6-v2` outputs normalized vectors. 
When vectors are normalized, dot product = cosine similarity.

Cosine similarity measures just the angle, not length, between vectors:
- cos(0) = 1.0 same direction
- cos(90) = 0.0 perpendicular (unrelated)
- cos(180) = -1.0 opposite (opposite meaning)


In practice, below 0 is rare.

## Creating the Embeddings

In [13]:
# load FAQ data
from ingest import load_faq_data

documents = load_faq_data()

In [21]:
documents[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 'doc_id': '9e508f2212'}

We make the choice to embed the question and answer together as a single text.

In [15]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [16]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [17]:
from tqdm.auto import tqdm

In [18]:
# chunk the dataset into batches
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    # model.encode is where the embeddings are generated
    batch_vectors = model.encode(batch) 
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [19]:
type(vectors)

list

## Vector Search

We have a list of 1350 vectors. We'll turn it into a 2-d array so we can perform matrix-vector multiplication, this is FASTER because
- Numpy matrices run C code
- looping vectors is just plain Python

In [22]:
import numpy as np
X = np.array(vectors)

In [ ]:
# show we have 1350 documents, each 384 dimensions
X.shape

(1350, 384)

In [24]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [25]:
# scores = [v_query.dot(X[i]) for i in range(len(X))]
# SLOW

# instead, to compute the dot product of the query vector with all document vectors
scores = X.dot(v_query)


The highest score is the most similar document.

In [26]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [27]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [28]:
# sorts lowest to highest
top5 = np.argsort(scores)[-5:]

In [29]:
top5 = top5[::-1] # reverse
top5

array([  2, 625, 907, 538,   7])

In [30]:
# you can also negate the scores first so you don't need to reverse
# top5 = np.argsort(-scores)[:5]

In [31]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [34]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

### Summary of Simple Vector Search

Simple vector search:
1. Embed documents
2. Embed query
3. Compute dot products against all documents.
4. Return the highest-scoring ones. 

We return 5 instead of 1 so the LLM can combine if needed, if the answer is spread.

A larger dataset requires filtering and ranking documents too. 

## Vector Search with Min Search

In [35]:
from minsearch import VectorSearch

In [36]:
# instead of a text index, create a vector search index
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

# fit on the embeddings AND documents

In [37]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

`vindex.search` did the same thing under the hood - computed the dot product between X (after filtering) and the query vector

In [38]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [39]:
# add a filter 
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## Using this Updated Vector Search in Our RAG Flow

In [ ]:
def rag(question):
    search_results = search(question) # WE'LL SWAP THIS PART OUT
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
# 1. OpenAI Client
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [41]:
# 2. Download and index data
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
# 3. Use RAGBase
from rag_helper import RAGBase

assistant = RAGBase(
    index=index, # we'll try with the traditional text index first
    llm_client=openai_client,
)

In [43]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

In [50]:
query2 = "I just found the class, is enrolllment possible?" # slightly different wording
assistant.rag(query2)

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [44]:
# 4. Subclass RAGBase, override search.

class RAGVector(RAGBase):

    # add one additional arg, embedder
    def __init__(self, embedder, **kwargs): 
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [45]:
# 5. Use RAGVector to test RAG with vector search instead of text search

vector_assistant = RAGVector(embedder=model, index=vindex, llm_client=openai_client)

In [51]:
vector_assistant.rag(query2)

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still open.'

## What are the cons of Vector Search with minsearch?

1. Rebuilds index on every startup
2. Everything in memory (not scalalable)
3. Brute force search (compares every document, exact NN search)

## Using sqlitesearch for persistence and ANN

`sqlitesearch` has 3 ANN modes:
1. `lsh`: up to 100k vectors, random hyperplane projection
2. `ivf`: 10K-500K vectors, K-means clustering
3. `hnsw`: 10K-1M+ vectors, proximity graph (highest recall)

In [ ]:
from sqlitesearch import VectorSearchIndex

# this will be our index using SQLiteSearch
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
vs_index.fit(vectors, documents)

In [54]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

In [55]:
results = vs_index.search(query_vector, num_results=5)

In [56]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [57]:
# Add filtering
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [58]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [59]:
vs_index.close()